# 11b. Causal Inference: Predict Delta Tau

**Role of this notebook.** Train the outcome nuisance model for later DML causal inference. The target is the adjacent-session change in choice-implied selectivity:

$$
\Delta \tau_i = \tau^{choice}_{i,t+1} - \tau^{choice}_{i,t}.
$$

The model estimates the predictable component of `delta_tau` from pre-treatment information known before session `t`:

$$
\widehat m(Z_t) \approx E[\Delta\tau \mid Z_t].
$$

The residualized outcome saved by this notebook is:

$$
\widetilde{\Delta\tau} = \Delta\tau - \widehat m(Z_t).
$$

**Steps.**

1. Load the 11a transition data and fixed structural objects.
2. Build compact pre-treatment features: baseline tau, belief summaries, user preference summaries, shifted history, time, and static controls.
3. Split transitions by `user_id` into train/validation/test, increasing the test share until test has at least 4,000 transitions.
4. Fit and tune a gradient-boosting nuisance model on train/validation.
5. Report train, validation, and test prediction metrics.
6. Save predictions, residualized `delta_tau`, fitted model, feature table, and diagnostics.

This notebook does **not** construct the treatment and does **not** run the final DML regression.


## Feature Set

All predictors are measured before session `t` or are persistent structural/user attributes. The feature set intentionally excludes `tau_t1`, `delta_tau`, session `t` outcomes, session `t+1` features, treatment variables, raw `user_id`, session length, and first-recommended-video category history such as `hist_cat_ema_complete_t`.

| Feature | Type | Description |
|---|---|---|
| `tau_t` | numeric | Choice-implied selectivity threshold estimated for session `t` in notebook 11a. |
| `belief_entropy_t` | numeric | Entropy of the pre-session belief vector over level-1 categories. |
| `belief_max_t` | numeric | Largest category probability in the pre-session belief vector. |
| `belief_top3_sum_t` | numeric | Sum of the three largest category probabilities in the pre-session belief vector. |
| `belief_unknown_share_t` | numeric | Belief probability assigned to the unknown level-1 category, if present. |
| `belief_utility_mean_t` | numeric | Belief-weighted mean of fixed structural utility means. |
| `belief_utility_std_t` | numeric | Belief-implied utility standard deviation from fixed `mu` and `sigma`. |
| `mu_mean_i` | numeric | Mean of user `i`'s fixed utility means across level-1 categories. |
| `mu_std_i` | numeric | Standard deviation of user `i`'s fixed utility means across categories. |
| `mu_max_i` | numeric | Maximum of user `i`'s fixed category utility means. |
| `mu_min_i` | numeric | Minimum of user `i`'s fixed category utility means. |
| `mu_top3_mean_i` | numeric | Mean of user `i`'s top three category utility means. |
| `sigma_mean_i` | numeric | Mean of user `i`'s fixed category utility standard deviations. |
| `sigma_std_i` | numeric | Standard deviation of user `i`'s fixed category utility standard deviations. |
| `hist_ema_watchratio_t` | numeric | Shifted user watch-ratio EMA at the first row of session `t`. |
| `hist_ema_y_complete_t` | numeric | Shifted user completion EMA at the first row of session `t`. |
| `hist_ema_y_long_t` | numeric | Shifted user long-watch EMA at the first row of session `t`. |
| `hist_ema_y_rewatch_t` | numeric | Shifted user rewatch EMA at the first row of session `t`. |
| `hist_ema_y_neg_t` | numeric | Shifted user negative/short-watch EMA at the first row of session `t`. |
| `hist_cat_entropy_l2_t` | numeric | Entropy of user history over level-2 categories before session `t`. |
| `hist_prev_sess_len_t` | numeric | Length of the previous completed session before session `t`. |
| `hist_intersess_gap_h_t` | numeric | Hours between previous session end and session `t` start. |
| `session_idx_within_burst` | numeric | True order of session `t` within all structural sessions for the user's current burst. |
| `ctx_hour_sin_t` | numeric | Sine transform of the hour at the first row of session `t`. |
| `ctx_hour_cos_t` | numeric | Cosine transform of the hour at the first row of session `t`. |
| `ctx_is_weekend_t` | numeric | Weekend indicator at the first row of session `t`. |
| `burst_id` | categorical | Dense activity burst identifier. |
| `u_user_active_degree` | categorical | Platform-provided user activity segment. |
| `u_register_days_range` | categorical | Binned account-age segment. |


In [1]:
from pathlib import Path
import json
import math

import joblib
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
DATA_DIR = ROOT / 'KuaiRec 2.0' / 'data' / 'processed'
OUT_DIR = ROOT / 'python_code_new' / 'outputs'

TRANSITIONS_PATH = OUT_DIR / 'causal_tau_choice_transitions.parquet'
SESSIONS_PATH = OUT_DIR / 'causal_tau_choice_sessions.parquet'
STRUCTURAL_THETA_PHI_PATH = OUT_DIR / 'structural_estimates_theta_phi.npz'
STRUCTURAL_ARRAYS_PATH = OUT_DIR / 'structural_estimation_arrays.npz'
STRUCTURAL_SESSIONS_PATH = OUT_DIR / 'structural_sessions.parquet'
PROCESSED_DATA_PATH = DATA_DIR / 'processed_data.parquet'

PREDICTIONS_PATH = OUT_DIR / 'causal_delta_tau_predictions.parquet'
METRICS_PATH = OUT_DIR / 'causal_delta_tau_prediction_metrics.json'
MODEL_PATH = OUT_DIR / 'causal_delta_tau_predictor.joblib'
FEATURE_TABLE_PATH = OUT_DIR / 'causal_delta_tau_feature_table.csv'

RANDOM_SEED = 20260628
TEST_MIN_TRANSITIONS = 4000
VALID_USER_SHARE = 0.15
INITIAL_TEST_USER_SHARE = 0.15
MAX_TEST_USER_SHARE = 0.35
TEST_SHARE_STEP = 0.05

for p in [TRANSITIONS_PATH, SESSIONS_PATH, STRUCTURAL_THETA_PHI_PATH, STRUCTURAL_ARRAYS_PATH, STRUCTURAL_SESSIONS_PATH, PROCESSED_DATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print('output dir:', OUT_DIR)


output dir: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs


## 1. Load Transition Target and Fixed Structural Objects

The target comes from notebook 11a:

$$
\Delta\tau = \tau_{t+1}^{choice} - \tau_t^{choice}.
$$

The user preference summaries are built from fixed notebook-06 structural estimates. They capture persistent user heterogeneity without using raw `user_id` as a predictor.


In [2]:
transitions = pd.read_parquet(TRANSITIONS_PATH)
tau_sessions = pd.read_parquet(SESSIONS_PATH)
theta_phi_npz = np.load(STRUCTURAL_THETA_PHI_PATH, allow_pickle=True)
arrays_npz = np.load(STRUCTURAL_ARRAYS_PATH, allow_pickle=True)

if 'delta_tau' not in transitions.columns:
    transitions['delta_tau'] = transitions['tau_t1'] - transitions['tau_t']

cat_ids = theta_phi_npz['cat_ids'].astype(np.int64)
reference_cat_idx = int(theta_phi_npz['reference_cat_idx'][0])
mu_by_user = theta_phi_npz['theta_b_cat'][None, :] + theta_phi_npz['theta_U'] @ theta_phi_npz['theta_V'].T
mu_by_user = mu_by_user - mu_by_user[:, [reference_cat_idx]]
sigmas_by_user = arrays_npz['sigmas_by_user'].astype(np.float64)

user_pref = pd.DataFrame({
    'user_idx_int': np.arange(mu_by_user.shape[0], dtype=np.int64),
    'mu_mean_i': mu_by_user.mean(axis=1),
    'mu_std_i': mu_by_user.std(axis=1),
    'mu_max_i': mu_by_user.max(axis=1),
    'mu_min_i': mu_by_user.min(axis=1),
    'mu_top3_mean_i': np.sort(mu_by_user, axis=1)[:, -3:].mean(axis=1),
    'sigma_mean_i': sigmas_by_user.mean(axis=1),
    'sigma_std_i': sigmas_by_user.std(axis=1),
})

print('transition rows:', f'{len(transitions):,}')
print('users with transitions:', transitions['user_id'].nunique())
display(transitions['delta_tau'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)


transition rows: 20,295
users with transitions: 1776


,count,mean,std,min,1%,5%,50%,95%,99%,max
delta_tau,20295.0,0.016306,4.871719,-25.989528,-13.34554,-9.509213,0.00476,9.610364,13.651976,25.261313


## 2. Construct Pre-Treatment Features

Belief summaries are computed from the pre-session belief vector saved by notebook 11a. History and static controls are taken from the first interaction row of session `t`, because these global history variables are shifted and represent information available before that row's outcome. Category-specific first-row history `hist_cat_ema_complete_t` is intentionally excluded because it conditions on the category of the first actually recommended video in session `t`.

The construction rule is:

$$
Z_t = \text{information known before or at the start of session } t.
$$

The feature matrix does not include realized session length, current-session outcomes, next-session features, treatment variables, raw user identifiers, or current-session category-specific first-row features. The session order variable is merged from the full structural session table before transition filtering.


In [3]:
belief_cols_t = sorted([c for c in transitions.columns if c.startswith('B_k_') and c.endswith('_t')])
if not belief_cols_t:
    raise ValueError('No B_k_*_t columns found in transition table')

belief_mat = transitions[belief_cols_t].to_numpy(dtype=np.float64)
transitions['belief_max_t'] = np.nanmax(belief_mat, axis=1)
transitions['belief_top3_sum_t'] = np.sort(belief_mat, axis=1)[:, -3:].sum(axis=1)

unknown_matches = np.where(cat_ids == -124)[0]
if len(unknown_matches):
    unknown_col = f'B_k_{int(unknown_matches[0]):02d}_t'
    transitions['belief_unknown_share_t'] = transitions[unknown_col].astype(float)
else:
    unknown_col = None
    transitions['belief_unknown_share_t'] = 0.0

# True order of session t within the full structural session table, before any adjacent-pair filtering.
structural_sessions_full = pd.read_parquet(STRUCTURAL_SESSIONS_PATH, columns=['user_id', 'burst_id', 'session'])
session_order_table = (
    structural_sessions_full
    .drop_duplicates(['user_id', 'burst_id', 'session'])
    .sort_values(['user_id', 'burst_id', 'session'], kind='mergesort')
    .reset_index(drop=True)
)
session_order_table['session_idx_within_burst'] = (
    session_order_table.groupby(['user_id', 'burst_id'], sort=False).cumcount().astype(np.int32)
)
transitions = transitions.merge(
    session_order_table,
    on=['user_id', 'burst_id', 'session'],
    how='left',
    validate='many_to_one',
)
if transitions['session_idx_within_burst'].isna().any():
    raise ValueError('Missing true session_idx_within_burst for some transitions')
transitions['session_idx_within_burst'] = transitions['session_idx_within_burst'].astype(np.int32)

# Sanity check: show that the true full-session order is not the compressed selected-transition order.
selected_order_proxy = (
    transitions.sort_values(['user_id', 'burst_id', 'session'])
    .groupby(['user_id', 'burst_id'], sort=False)
    .cumcount()
    .astype(np.int32)
)
session_order_check = transitions[['user_id', 'burst_id', 'session', 'session_idx_within_burst']].copy()
session_order_check['selected_transition_cumcount_wrong'] = selected_order_proxy.to_numpy()
session_order_check['differs_from_wrong_compressed_order'] = (
    session_order_check['session_idx_within_burst'] != session_order_check['selected_transition_cumcount_wrong']
)
print('session order construction: full structural session table, grouped by user_id and burst_id')
print('share differing from compressed selected-transition cumcount:', float(session_order_check['differs_from_wrong_compressed_order'].mean()))
display(session_order_check[session_order_check['differs_from_wrong_compressed_order']].head(10))

# Load only compact columns needed for pre-session controls. Exclude hist_cat_ema_complete because it is tied to the
# category of the first actually recommended video in session t.
history_cols = [
    'user_id', 'burst_id', 'session', 'time', 'video_id',
    'hist_ema_watchratio', 'hist_ema_y_complete', 'hist_ema_y_long', 'hist_ema_y_rewatch', 'hist_ema_y_neg',
    'hist_cat_entropy_l2', 'hist_prev_sess_len', 'hist_intersess_gap_h',
    'ctx_hour_sin', 'ctx_hour_cos', 'ctx_is_weekend',
    'u_user_active_degree', 'u_register_days_range',
]
processed = pd.read_parquet(PROCESSED_DATA_PATH, columns=history_cols)
selected_keys = transitions[['user_id', 'burst_id', 'session']].drop_duplicates()
processed = processed.merge(selected_keys, on=['user_id', 'burst_id', 'session'], how='inner')

first_rows = (
    processed
    .sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort')
    .groupby(['user_id', 'burst_id', 'session'], observed=True, as_index=False)
    .first()
)

rename_map = {
    'hist_ema_watchratio': 'hist_ema_watchratio_t',
    'hist_ema_y_complete': 'hist_ema_y_complete_t',
    'hist_ema_y_long': 'hist_ema_y_long_t',
    'hist_ema_y_rewatch': 'hist_ema_y_rewatch_t',
    'hist_ema_y_neg': 'hist_ema_y_neg_t',
    'hist_cat_entropy_l2': 'hist_cat_entropy_l2_t',
    'hist_prev_sess_len': 'hist_prev_sess_len_t',
    'hist_intersess_gap_h': 'hist_intersess_gap_h_t',
    'ctx_hour_sin': 'ctx_hour_sin_t',
    'ctx_hour_cos': 'ctx_hour_cos_t',
    'ctx_is_weekend': 'ctx_is_weekend_t',
}
first_rows = first_rows.rename(columns=rename_map)
first_rows['u_user_active_degree'] = first_rows['u_user_active_degree'].astype('object').astype(str)
first_rows['u_register_days_range'] = first_rows['u_register_days_range'].astype('object').astype(str)

feature_frame = transitions.merge(
    first_rows.drop(columns=['time', 'video_id']),
    on=['user_id', 'burst_id', 'session'],
    how='left',
    validate='many_to_one',
).merge(
    user_pref,
    on='user_idx_int',
    how='left',
    validate='many_to_one',
)

print('processed first-row controls:', first_rows.shape)
print('feature frame:', feature_frame.shape)


session order construction: full structural session table, grouped by user_id and burst_id
share differing from compressed selected-transition cumcount: 0.6456270017245627


,user_id,burst_id,session,session_idx_within_burst,selected_transition_cumcount_wrong,differs_from_wrong_compressed_order
11,3,2,20,8,6,True
12,3,2,21,9,7,True
13,3,2,22,10,8,True
14,3,2,23,11,9,True
15,3,2,24,12,10,True
16,3,2,28,16,11,True
17,3,2,29,17,12,True
19,7,1,10,6,0,True
20,7,1,11,7,1,True
24,7,2,22,5,3,True


processed first-row controls: (20295, 18)
feature frame: (20295, 87)


## 3. Final Feature Lists

The model uses numerical features with median imputation and categorical features with one-hot encoding. Any requested feature that cannot be constructed is dropped and recorded in the diagnostics.


In [4]:
requested_numeric_features = [
    'tau_t',
    'belief_entropy_t', 'belief_max_t', 'belief_top3_sum_t', 'belief_unknown_share_t',
    'belief_utility_mean_t', 'belief_utility_std_t',
    'mu_mean_i', 'mu_std_i', 'mu_max_i', 'mu_min_i', 'mu_top3_mean_i', 'sigma_mean_i', 'sigma_std_i',
    'hist_ema_watchratio_t', 'hist_ema_y_complete_t', 'hist_ema_y_long_t', 'hist_ema_y_rewatch_t', 'hist_ema_y_neg_t',
    'hist_cat_entropy_l2_t', 'hist_prev_sess_len_t', 'hist_intersess_gap_h_t',
    'session_idx_within_burst', 'ctx_hour_sin_t', 'ctx_hour_cos_t', 'ctx_is_weekend_t',
]
requested_categorical_features = ['burst_id', 'u_user_active_degree', 'u_register_days_range']

numeric_features = [c for c in requested_numeric_features if c in feature_frame.columns]
categorical_features = [c for c in requested_categorical_features if c in feature_frame.columns]
dropped_features = [c for c in requested_numeric_features + requested_categorical_features if c not in feature_frame.columns]
intentionally_dropped_features = ['hist_cat_ema_complete_t']

for c in numeric_features:
    feature_frame[c] = pd.to_numeric(feature_frame[c], errors='coerce')
for c in categorical_features:
    feature_frame[c] = feature_frame[c].astype('object').where(feature_frame[c].notna(), '__MISSING__').astype(str)

feature_descriptions = {
    'tau_t': 'Choice-implied selectivity threshold for session t from notebook 11a.',
    'belief_entropy_t': 'Entropy of the pre-session belief vector over level-1 categories.',
    'belief_max_t': 'Largest pre-session belief probability across categories.',
    'belief_top3_sum_t': 'Sum of the three largest pre-session category probabilities.',
    'belief_unknown_share_t': 'Belief probability on the unknown level-1 category; zero if the category is absent.',
    'belief_utility_mean_t': 'Belief-weighted mean of fixed structural utility means.',
    'belief_utility_std_t': 'Belief-implied standard deviation of utility using fixed mu and sigma.',
    'mu_mean_i': 'Mean of fixed category utility means for user i.',
    'mu_std_i': 'Standard deviation of fixed category utility means for user i.',
    'mu_max_i': 'Maximum fixed category utility mean for user i.',
    'mu_min_i': 'Minimum fixed category utility mean for user i.',
    'mu_top3_mean_i': 'Mean of the top three fixed category utility means for user i.',
    'sigma_mean_i': 'Mean fixed category utility standard deviation for user i.',
    'sigma_std_i': 'Standard deviation of fixed category utility standard deviations for user i.',
    'hist_ema_watchratio_t': 'Shifted watch-ratio EMA before the first interaction of session t.',
    'hist_ema_y_complete_t': 'Shifted completion EMA before the first interaction of session t.',
    'hist_ema_y_long_t': 'Shifted long-watch EMA before the first interaction of session t.',
    'hist_ema_y_rewatch_t': 'Shifted rewatch EMA before the first interaction of session t.',
    'hist_ema_y_neg_t': 'Shifted negative/short-watch EMA before the first interaction of session t.',
    'hist_cat_entropy_l2_t': 'Entropy of past level-2 category history before session t.',
    'hist_prev_sess_len_t': 'Length of the previous completed session before session t.',
    'hist_intersess_gap_h_t': 'Hours between previous session end and session t start.',
    'session_idx_within_burst': 'True order of session t within all structural sessions for the user-burst.',
    'ctx_hour_sin_t': 'Sine of hour-of-day at the first interaction of session t.',
    'ctx_hour_cos_t': 'Cosine of hour-of-day at the first interaction of session t.',
    'ctx_is_weekend_t': 'Weekend indicator at the first interaction of session t.',
    'burst_id': 'Dense activity burst identifier, one-hot encoded.',
    'u_user_active_degree': 'Platform-provided user activity segment, one-hot encoded.',
    'u_register_days_range': 'Binned account-age segment, one-hot encoded.',
}
feature_table = pd.DataFrame([
    {'feature': c, 'type': 'numeric' if c in numeric_features else 'categorical', 'description': feature_descriptions.get(c, '')}
    for c in numeric_features + categorical_features
])
feature_table.to_csv(FEATURE_TABLE_PATH, index=False)

display(feature_table)
print('numeric features:', len(numeric_features))
print('categorical features:', len(categorical_features))
print('dropped features:', dropped_features)
print('intentionally dropped features:', intentionally_dropped_features)


,feature,type,description
0,tau_t,numeric,Choice-implied selectivity threshold for sessi...
1,belief_entropy_t,numeric,Entropy of the pre-session belief vector over ...
2,belief_max_t,numeric,Largest pre-session belief probability across ...
3,belief_top3_sum_t,numeric,Sum of the three largest pre-session category ...
4,belief_unknown_share_t,numeric,Belief probability on the unknown level-1 cate...
5,belief_utility_mean_t,numeric,Belief-weighted mean of fixed structural utili...
6,belief_utility_std_t,numeric,Belief-implied standard deviation of utility u...
7,mu_mean_i,numeric,Mean of fixed category utility means for user i.
8,mu_std_i,numeric,Standard deviation of fixed category utility m...
9,mu_max_i,numeric,Maximum fixed category utility mean for user i.


numeric features: 26
categorical features: 3
dropped features: []
intentionally dropped features: ['hist_cat_ema_complete_t']


## 4. User-Level Train, Validation, and Test Split

The split is by `user_id`, not by transition row. This prevents the nuisance model from training on one transition from a user and being evaluated on another transition from the same user.

The default target is roughly 70% train users, 15% validation users, and 15% test users. If 15% test users gives fewer than 4,000 test transitions, the notebook increases the test-user share until the test set is large enough.


In [5]:
def assign_user_split(frame, seed=RANDOM_SEED):
    users = np.array(sorted(frame['user_id'].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(users)
    n_users = len(users)

    chosen = None
    for test_share in np.arange(INITIAL_TEST_USER_SHARE, MAX_TEST_USER_SHARE + 1e-9, TEST_SHARE_STEP):
        n_test = int(math.ceil(n_users * float(test_share)))
        n_valid = int(math.ceil(n_users * VALID_USER_SHARE))
        test_users = set(users[:n_test])
        valid_users = set(users[n_test:n_test + n_valid])
        train_users = set(users[n_test + n_valid:])
        split = np.where(frame['user_id'].isin(test_users), 'test', np.where(frame['user_id'].isin(valid_users), 'valid', 'train'))
        test_rows = int((split == 'test').sum())
        chosen = (test_share, train_users, valid_users, test_users, split, test_rows)
        if test_rows >= TEST_MIN_TRANSITIONS:
            break
    return chosen

test_share, train_users, valid_users, test_users, split, test_rows = assign_user_split(feature_frame)
feature_frame['split'] = split

split_summary = (
    feature_frame.groupby('split')
    .agg(n_obs=('delta_tau', 'size'), n_users=('user_id', 'nunique'), target_mean=('delta_tau', 'mean'), target_std=('delta_tau', 'std'))
    .reindex(['train', 'valid', 'test'])
)

print('chosen test user share:', test_share)
print('test transitions:', test_rows)
display(split_summary)

if test_rows < TEST_MIN_TRANSITIONS:
    raise ValueError(f'test split has only {test_rows} transitions, below {TEST_MIN_TRANSITIONS}')


chosen test user share: 0.2
test transitions: 4069


,n_obs,n_users,target_mean,target_std
split,,,,
train,13158,1153,-0.004870,4.912699
valid,3068,267,0.015246,4.740992
test,4069,356,0.085582,4.836284


## 5. Train and Tune the Outcome Nuisance Model

The main nuisance model is gradient boosting for tabular regression. The model is selected by validation RMSE over a small hyperparameter grid. The selected model is then used to predict every split.

The fitted function is:

$$
\widehat m(Z_t)=\widehat E[\Delta\tau\mid Z_t].
$$


In [6]:
target_col = 'delta_tau'
feature_cols = numeric_features + categorical_features
assert 'hist_cat_ema_complete_t' not in feature_cols
assert 'user_id' not in feature_cols

train_mask = feature_frame['split'].eq('train')
valid_mask = feature_frame['split'].eq('valid')
test_mask = feature_frame['split'].eq('test')

X_train = feature_frame.loc[train_mask, feature_cols]
y_train = feature_frame.loc[train_mask, target_col].to_numpy(dtype=np.float64)
X_valid = feature_frame.loc[valid_mask, feature_cols]
y_valid = feature_frame.loc[valid_mask, target_col].to_numpy(dtype=np.float64)
X_test = feature_frame.loc[test_mask, feature_cols]
y_test = feature_frame.loc[test_mask, target_col].to_numpy(dtype=np.float64)

try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
            ('onehot', onehot),
        ]), categorical_features),
    ],
    sparse_threshold=0.0,
)

param_grid = [
    {'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.01},
    {'max_iter': 500, 'learning_rate': 0.04, 'max_leaf_nodes': 31, 'l2_regularization': 0.01},
    {'max_iter': 500, 'learning_rate': 0.05, 'max_leaf_nodes': 63, 'l2_regularization': 0.05},
    {'max_iter': 800, 'learning_rate': 0.03, 'max_leaf_nodes': 31, 'l2_regularization': 0.05},
]

def make_model(params):
    reg = HistGradientBoostingRegressor(
        loss='squared_error',
        random_state=RANDOM_SEED,
        early_stopping=False,
        **params,
    )
    return Pipeline(steps=[('preprocess', preprocess), ('model', reg)])

def rmse(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

tuning_records = []
best_model = None
best_params = None
best_valid_rmse = np.inf

for params in param_grid:
    model = make_model(params)
    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)
    rec = dict(params)
    rec['valid_rmse'] = rmse(y_valid, pred_valid)
    rec['valid_mae'] = float(mean_absolute_error(y_valid, pred_valid))
    rec['valid_r2'] = float(r2_score(y_valid, pred_valid))
    tuning_records.append(rec)
    if rec['valid_rmse'] < best_valid_rmse:
        best_valid_rmse = rec['valid_rmse']
        best_params = dict(params)
        best_model = model

tuning_df = pd.DataFrame(tuning_records).sort_values('valid_rmse')
display(tuning_df)
print('best params:', best_params)


,max_iter,learning_rate,max_leaf_nodes,l2_regularization,valid_rmse,valid_mae,valid_r2
0,300,0.05,31,0.01,3.754083,2.630701,0.372793
3,800,0.03,31,0.05,3.773587,2.649504,0.366259
2,500,0.05,63,0.05,3.784840,2.662278,0.362474
1,500,0.04,31,0.01,3.786279,2.652711,0.361989


best params: {'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.01}


## 6. Evaluate Predictions and Residualize Outcome

The residualized outcome is saved for all splits:

$$
\Delta\tau_{resid}=\Delta\tau-\widehat m(Z_t).
$$

For later DML inference, the important residuals are the test-split residuals because those users were not used for nuisance fitting or tuning.


In [7]:
def pearson_corr(y, pred):
    if len(y) < 2 or np.std(y) == 0 or np.std(pred) == 0:
        return np.nan
    return float(stats.pearsonr(y, pred).statistic)

def spearman_corr(y, pred):
    if len(y) < 2 or np.std(y) == 0 or np.std(pred) == 0:
        return np.nan
    return float(stats.spearmanr(y, pred).statistic)

def eval_split(name, mask):
    X = feature_frame.loc[mask, feature_cols]
    y = feature_frame.loc[mask, target_col].to_numpy(dtype=np.float64)
    pred = best_model.predict(X)
    return pred, {
        'split': name,
        'n_obs': int(mask.sum()),
        'n_users': int(feature_frame.loc[mask, 'user_id'].nunique()),
        'target_mean': float(np.mean(y)),
        'target_std': float(np.std(y, ddof=1)),
        'pred_mean': float(np.mean(pred)),
        'pred_std': float(np.std(pred, ddof=1)),
        'rmse': rmse(y, pred),
        'mae': float(mean_absolute_error(y, pred)),
        'r2': float(r2_score(y, pred)),
        'pearson': pearson_corr(y, pred),
        'spearman': spearman_corr(y, pred),
    }

feature_frame['pred_delta_tau'] = np.nan
metrics = []
for split_name, mask in [('train', train_mask), ('valid', valid_mask), ('test', test_mask)]:
    pred, rec = eval_split(split_name, mask)
    feature_frame.loc[mask, 'pred_delta_tau'] = pred
    metrics.append(rec)

feature_frame['delta_tau_resid'] = feature_frame['delta_tau'] - feature_frame['pred_delta_tau']
metrics_df = pd.DataFrame(metrics)
display(metrics_df)

target_summary = feature_frame['delta_tau'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict()
resid_summary = feature_frame.groupby('split')['delta_tau_resid'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(resid_summary)


,split,n_obs,n_users,target_mean,target_std,pred_mean,pred_std,rmse,mae,r2,pearson,spearman
0,train,13158,1153,-0.004870,4.912699,-0.004870,3.393229,2.767720,1.941715,0.682578,0.839469,0.580014
1,valid,3068,267,0.015246,4.740992,0.108736,3.200578,3.754083,2.630701,0.372793,0.613939,0.429200
2,test,4069,356,0.085582,4.836284,-0.002864,3.398940,3.839401,2.742445,0.369609,0.614593,0.465963


,count,mean,std,min,1%,5%,50%,95%,99%,max
split,,,,,,,,,,
test,4069.0,8.844591e-02,3.838854,-14.635182,-9.609496,-6.993567,0.131761,6.727718,9.207921,16.925554
train,13158.0,1.218971e-07,2.767826,-12.779061,-8.062929,-5.170842,0.120603,4.438708,6.582764,13.800637
valid,3068.0,-9.348976e-02,3.753530,-13.904421,-9.868200,-7.139608,0.073526,6.289931,8.512950,18.159167


## 7. Save Predictions, Model, and Diagnostics

The prediction file contains the variables needed by later DML notebooks:

| Column | Meaning |
|---|---|
| `split` | user-level train, validation, or test assignment |
| `delta_tau` | observed change in choice-implied selectivity |
| `pred_delta_tau` | nuisance prediction `E[delta_tau | Z_t]` |
| `delta_tau_resid` | residualized outcome for later DML |

The fitted preprocessing pipeline and gradient-boosting model are saved together with `joblib`.


In [8]:
prediction_cols = [
    'user_id', 'user_idx_int', 'burst_id', 'session', 'next_session', 'split',
    'tau_t', 'tau_t1', 'delta_tau', 'pred_delta_tau', 'delta_tau_resid',
    'tau_model_old_t', 'tau_model_old_t1', 'delta_tau_model_old',
]
for col in prediction_cols:
    if col not in feature_frame.columns:
        raise KeyError(f'missing prediction output column: {col}')

predictions = feature_frame[prediction_cols].sort_values(['split', 'user_id', 'burst_id', 'session']).reset_index(drop=True)
predictions.to_parquet(PREDICTIONS_PATH, index=False)
joblib.dump(best_model, MODEL_PATH)

metrics_payload = {
    'random_seed': RANDOM_SEED,
    'target': target_col,
    'target_definition': 'delta_tau = tau_t1 - tau_t',
    'n_rows': int(len(feature_frame)),
    'n_users': int(feature_frame['user_id'].nunique()),
    'split_by': 'user_id',
    'chosen_test_user_share': float(test_share),
    'test_min_transitions': int(TEST_MIN_TRANSITIONS),
    'split_summary': split_summary.reset_index().to_dict(orient='records'),
    'metrics': metrics_df.to_dict(orient='records'),
    'target_summary': target_summary,
    'feature_columns_used': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'dropped_features': dropped_features,
    'intentionally_dropped_features': intentionally_dropped_features,
    'hist_cat_ema_complete_t_note': 'Dropped because first-row category history conditions on the category of the first actually recommended video in session t.',
    'session_order_construction': 'session_idx_within_burst is computed from the full structural_sessions.parquet table before adjacent-pair filtering, grouped by user_id and burst_id.',
    'session_order_check_share_differs_from_wrong_compressed_order': float(session_order_check['differs_from_wrong_compressed_order'].mean()),
    'test_metrics': next((r for r in metrics_df.to_dict(orient='records') if r.get('split') == 'test'), None),
    'feature_table': feature_table.to_dict(orient='records'),
    'best_hyperparameters': best_params,
    'tuning_records': tuning_df.to_dict(orient='records'),
    'excluded_features_policy': [
        'raw user_id excluded because split is by user',
        'tau_t1 and delta_tau excluded from predictors',
        'session lengths excluded from predictors',
        'hist_cat_ema_complete_t excluded because it conditions on actual first recommended category in session t',
        'session t outcomes and session t+1 features excluded',
        'treatment variables are not constructed in this notebook',
    ],
    'input_paths': {
        'transitions': str(TRANSITIONS_PATH),
        'tau_sessions': str(SESSIONS_PATH),
        'processed_data': str(PROCESSED_DATA_PATH),
        'structural_theta_phi': str(STRUCTURAL_THETA_PHI_PATH),
        'structural_arrays': str(STRUCTURAL_ARRAYS_PATH),
        'structural_sessions': str(STRUCTURAL_SESSIONS_PATH),
    },
    'output_paths': {
        'predictions': str(PREDICTIONS_PATH),
        'metrics': str(METRICS_PATH),
        'model': str(MODEL_PATH),
        'feature_table': str(FEATURE_TABLE_PATH),
    },
}
METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2, allow_nan=True))

print('saved predictions:', PREDICTIONS_PATH)
print('saved metrics:', METRICS_PATH)
print('saved model:', MODEL_PATH)
print('saved feature table:', FEATURE_TABLE_PATH)


saved predictions: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_delta_tau_predictions.parquet
saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_delta_tau_prediction_metrics.json
saved model: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_delta_tau_predictor.joblib
saved feature table: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_delta_tau_feature_table.csv
